# Exploratory Data Analysis: UCI Online Retail Dataset

**Course:** CSCE 676 Data Mining  
**Purpose:** Data basics, data collection/cleaning, bias considerations, and initial insights for frequent itemsets, association rules, and sequential pattern mining.

## 1. Setup and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
sns.set_style('whitegrid')

# Path to data (adjust if your file is elsewhere)
DATA_DIR = os.path.join('..', 'data')
CSV_PATH = os.path.join(DATA_DIR, 'Online Retail.csv')

# If Excel was downloaded instead:
XLSX_PATH = os.path.join(DATA_DIR, 'Online Retail.xlsx')

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH, encoding='utf-8')
elif os.path.exists(XLSX_PATH):
    df = pd.read_excel(XLSX_PATH)
else:
    raise FileNotFoundError(f'Place Online Retail.csv or Online Retail.xlsx in {os.path.abspath(DATA_DIR)}')

print('Shape:', df.shape)
df.head(10)

## 2. Data basics: dtypes, missing values, duplicates

In [ ]:
print('\n--- dtypes ---')
print(df.dtypes)
print('\n--- Missing values ---')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing': missing, 'pct': missing_pct})
print('\n--- Duplicate rows ---')
print('Duplicate rows:', df.duplicated().sum())

## 3. Data cleaning: cancellations, returns, invalid quantities

In [ ]:
# Cancelled invoices (often prefixed with 'C')
if df['InvoiceNo'].dtype == object:
    cancelled = df['InvoiceNo'].astype(str).str.upper().str.startswith('C')
    print('Cancelled invoices (rows):', cancelled.sum())

# Negative quantity (returns)
neg_qty = (df['Quantity'] <= 0).sum()
print('Rows with Quantity <= 0:', neg_qty)

# Optional: create a cleaned copy for basket analysis (keep only valid transactions)
df_clean = df.copy()
if df['InvoiceNo'].dtype == object:
    df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.upper().str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]
print('\nCleaned shape:', df_clean.shape)

## 4. Distribution of basket sizes (items per transaction)

In [ ]:
basket_sizes = df_clean.groupby('InvoiceNo').size()
print('Basket size - describe:')
print(basket_sizes.describe())
print('\nMedian basket size:', basket_sizes.median())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
basket_sizes.hist(bins=50, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Items per transaction')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of basket sizes')
# Zoom into typical range
basket_sizes_clip = basket_sizes.clip(upper=50)
basket_sizes_clip.hist(bins=50, ax=axes[1], edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Items per transaction (capped at 50)')
axes[1].set_ylabel('Count')
axes[1].set_title('Basket sizes (zoomed)')
plt.tight_layout()
plt.show()

## 5. Frequency of top items and sparsity

In [ ]:
n_invoices = df_clean['InvoiceNo'].nunique()
item_counts = df_clean.groupby('StockCode')['InvoiceNo'].nunique()  # transactions containing each item
item_freq = (item_counts / n_invoices * 100).sort_values(ascending=False)

print('Unique items (StockCode):', len(item_counts))
print('Unique transactions:', n_invoices)
print('\nTop 15 items by % of transactions:')
print(item_freq.head(15))

# How many items appear in <1% of transactions?
rare = (item_freq < 1).sum()
print(f'\nItems appearing in <1% of transactions: {rare} ({rare/len(item_freq)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 4))
item_freq.head(20).plot(kind='barh', ax=ax)
ax.set_xlabel('% of transactions')
ax.set_title('Top 20 items by transaction frequency')
plt.tight_layout()
plt.show()

## 6. Sparsity of item co-occurrence (motivation for support thresholds)

In [ ]:
# Pairwise: how many transaction pairs share at least 2 items? (conceptually: sparsity)
baskets = df_clean.groupby('InvoiceNo')['StockCode'].apply(list).to_dict()
n_baskets = len(baskets)
n_items = df_clean['StockCode'].nunique()
total_pairs_possible = n_items * (n_items - 1) // 2
print(f'Transactions: {n_baskets}, Unique items: {n_items}')
print(f'Possible item pairs: {total_pairs_possible}')

# Count how many baskets contain each item (for support)
from collections import Counter
item_support = Counter()
for inv, items in baskets.items():
    for i in set(items):
        item_support[i] += 1
support_pct = np.array([item_support[i] / n_baskets * 100 for i in item_support])
print(f'\nSupport distribution: min={support_pct.min():.4f}%, max={support_pct.max():.2f}%, median={np.median(support_pct):.4f}%')
print('→ High sparsity: most items appear in very few baskets; motivates minimum support in Apriori/FP-Growth.')

## 7. Temporal: invoice dates and gaps (for sequential pattern mining)

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
print('Date range:', df_clean['InvoiceDate'].min(), 'to', df_clean['InvoiceDate'].max())
print('Transactions per month:')
monthly = df_clean.groupby(df_clean['InvoiceDate'].dt.to_period('M'))['InvoiceNo'].nunique()
print(monthly)

monthly.plot(kind='bar', figsize=(10, 4))
plt.xlabel('Month')
plt.ylabel('Number of transactions')
plt.title('Transaction volume over time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Bias considerations

In [ ]:
print('--- Geographic (Country) ---')
print(df_clean['Country'].value_counts().head(10))
print('\n--- CustomerID coverage ---')
with_id = df_clean['CustomerID'].notna().sum()
print(f'Rows with CustomerID: {with_id} ({with_id/len(df_clean)*100:.1f}%)')
print('\nBias notes: Data is UK-focused; many rows lack CustomerID (cannot attribute to repeat customers).')

## 9. Initial observations and direction

- **Observation:** Most items appear in fewer than 1% of transactions → high support thresholds may miss meaningful rules; low support yields many noisy rules.
- **Hypothesis:** Sequential patterns (e.g., with time windows per customer) may reveal structure that unordered frequent itemsets miss.
- **Next steps:** (1) Run Apriori/FP-Growth at several support levels; (2) build customer-level sequences and run sequential pattern mining; (3) compare rule quality and interpretability.